In [1]:
import os
import pandas as pd
import logfire
from dotenv import load_dotenv


In [2]:
import logging
import logfire
from dotenv import load_dotenv

# Initialization
load_dotenv(override=True)

# 1. Keep Pydantic absolutely quiet
logging.getLogger('pydantic').setLevel(logging.ERROR)

# 2. Let Logfire speak at INFO level so you see links and status
logging.getLogger('logfire').setLevel(logging.INFO)

# 3. The "Clean Engine" configuration
# Configure Logfire; Pydantic noise is controlled via instrument_pydantic(record='off') below
logfire.configure()

# 4. Final safety toggle
logfire.instrument_pydantic(record='off')

Logfire project URL: ]8;id=507735;https://logfire-eu.pydantic.dev/grospi/agenticdataanalyst\https://logfire-eu.pydantic.dev/grospi/agenticdataanalyst]8;;\

In [3]:

def validate_environment():
    expected_keys = [
        "OPENAI_API_KEY", 
        "ANTHROPIC_API_KEY", 
        "GOOGLE_API_KEY", 
        "GROQ_API_KEY"
    ]
    
    status = []
    for key in expected_keys:
        val = os.getenv(key)
        if val:
            status.append(f"✅ {key[:key.find('_')]:<10} : Found ({val[:7]}...)")
        else:
            status.append(f"❌ {key[:key.find('_')]:<10} : Missing")
            
    print("--- Environment Validation ---")
    print("\n".join(status))
    print(f"\n✅ Pandas version: {pd.__version__}")
    print("✅ Logfire: Instrumented and Ready")
    print("✅ Pydantic-AI: Tracing Enabled")

if __name__ == "__main__":
    validate_environment()

--- Environment Validation ---
✅ OPENAI     : Found (sk-proj...)
✅ ANTHROPIC  : Found (sk-ant-...)
✅ GOOGLE     : Found (AIzaSyD...)
✅ GROQ       : Found (gsk_Vo9...)

✅ Pandas version: 3.0.0
✅ Logfire: Instrumented and Ready
✅ Pydantic-AI: Tracing Enabled


In [ ]:
import os
from pathlib import Path
from pydantic import BaseModel, Field, model_validator
from typing import List, Literal, Optional
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

def load_prompt(name: str) -> str:
    return Path(f"prompts/{name}.txt").read_text(encoding="utf-8").strip()

# Strategy models: planning and quality gates
class AnalysisStrategy(BaseModel):
    run_outlier_detection: bool
    target_columns_for_outliers: List[str]
    run_correlation_matrix: bool
    business_hypothesis: str
    primary_questions: List[str]
    reasoning: str
    id_like_columns: List[str]
    excluded_from_profiling: List[str]
    correlation_columns: List[str]

class CriticalFinding(BaseModel):
    finding_type: Literal[
        "id_column_misuse",        # Correlation on ID columns
        "correlation_causation_error", # Causation without proof
        "unsupported_claim",        # Claim not in data
        "missing_data_inflation",   # "Need more data" syndrome
        "outlier_misinterpretation", # Wrong interpretation of outliers
        "redundancy",               # Duplicate insights across sections
        "weak_signal_overstatement", 
        "missing_limitations"         
    ]
    affected_insight: str
    reason: str
    severity: Literal["block", "warn"]

class CriticReport(BaseModel):
    approved: bool
    score: float
    findings: List[CriticalFinding]
    revision_instructions: Optional[str] = None

    @model_validator(mode='after')
    def enforce_score_consistency(self) -> 'CriticReport':
        """
        Ensures that the score reflects the severity of the findings.
        """
        # Identify if any critical failures exist
        has_block = any(f.severity == "block" for f in self.findings)
        
        # Reset score to 0.0 if a BLOCK-level error is present
        if has_block and self.score > 0.3:
            self.score = 0.0
            
        # Limit the maximum score for unapproved reports
        if not self.approved and self.score > 0.5:
            self.score = min(self.score, 0.4)

        # Cap approved reports at 0.85 unless zero findings
        if self.approved and self.findings and self.score > 0.85:
            self.score = 0.85    
        return self

class AnalystOutput(BaseModel):
    """
    Structured output for the Senior Analyst to ensure deep reasoning.
    """
    internal_thought_process: List[str] = Field(..., description="Step-by-step reasoning and mathematical checks.")
    hypothesis_validation: str = Field(..., description="Confirmation or rejection of the CDO's hypothesis.")
    final_report_markdown: str = Field(..., description="The final business-centric report.")

class SheetAnalysisPlan(BaseModel):
    """Per-sheet plan within a multi-sheet mission."""
    sheet_name: str
    should_profile: bool
    id_like_columns: List[str]
    excluded_from_profiling: List[str]
    correlation_columns: List[str]
    primary_focus: str 

class JoinInstruction(BaseModel):
    """A concrete join to execute before profiling."""
    left_sheet: str
    right_sheet: str
    on_column: str
    join_type: Literal["inner", "left", "outer"]
    result_name: str 

class MultiSheetStrategy(BaseModel):
    """CDO's full plan for a multi-sheet mission."""
    business_hypothesis: str
    primary_questions: List[str]
    analysis_mode: Literal["separate", "joined", "both"]
    per_sheet_plans: List[SheetAnalysisPlan]
    joins_to_execute: List[JoinInstruction]
    cross_sheet_hypothesis: str 
    reasoning: str


class SlideContent(BaseModel):
    slide_type: Literal[
        "title",
        "hypothesis",
        "finding",      # One per key finding section
        "limitation",   # Null rate / data honesty flags
        "conclusion"
    ]
    title: str
    body_bullets: List[str] = Field(default_factory=list)
    metric_callout: Optional[str] = None   # e.g. "r = -0.338" rendered large
    speaker_notes: str = ""

class PresentationSpec(BaseModel):
    deck_title: str
    subtitle: str                          # e.g. "Titanic Survival Analysis"
    hypothesis_verdict: Literal["CONFIRMED", "PARTIALLY CONFIRMED", "REJECTED"]
    slides: List[SlideContent]
    
    # Carry-through metadata for footer/branding
    critic_score: float
    analysis_mode: str


# 2. Infrastructure Setup
openrouter_provider = OpenAIProvider(
    base_url='https://openrouter.ai/api/v1',
    api_key=os.getenv('OPENROUTER_API_KEY')
)

# 3. Agent 1: The Orchestrator (Chief Data Officer)
# Defines the strategic plan and enforces ID-column exclusion constraints
orchestrator_agent = Agent(
    OpenAIChatModel('anthropic/claude-3.5-sonnet', provider=openrouter_provider),
    output_type=AnalysisStrategy,
    system_prompt=load_prompt("cdo_strategy")
)

# 4. Agent 2: Senior Data Analyst (The Reporting Specialist)
# Synthesizes data profile into business-centric narrative using the CDO's hypothesis

analyst_agent = Agent(
    OpenAIChatModel('openai/gpt-4o', provider=openrouter_provider),
    output_type=AnalystOutput, # Corrected to match your notebook's syntax
    system_prompt=load_prompt("analyst_synthesis")
) 

# 5. Agent 3: Ruthless Auditor (The Quality Critic)
# Validates the narrative against raw data to ensure accuracy and logical integrity
critic_agent = Agent(
    OpenAIChatModel('anthropic/claude-3.5-sonnet', provider=openrouter_provider),
    output_type=CriticReport,
    system_prompt=load_prompt("critic_auditor")
)


multi_orchestrator_agent = Agent(
    OpenAIChatModel('anthropic/claude-3.5-sonnet', provider=openrouter_provider),
    output_type=MultiSheetStrategy,
    system_prompt=load_prompt("cdo_multi_sheet"),
)

formatter_agent = Agent(
    OpenAIChatModel('anthropic/claude-3.5-sonnet', provider=openrouter_provider),
    output_type=PresentationSpec,
    system_prompt=load_prompt("formatter_agent")
)   

In [ ]:
import asyncio
import json
import os
import sys
import pandas as pd
import logfire
from pathlib import Path
from typing import Dict, cast

# Add static module directories to the Python search path
for _p in ("generated_modules", "lib"):
    _resolved = str(Path(_p).resolve())
    if _resolved not in sys.path:
        sys.path.insert(0, _resolved)

from data_profiler import ProfilerReport, get_full_profile
from data_discovery_lib import run_dataset_discovery
from path_utils import resolve_file_path

async def _profile_sheet_async(df: pd.DataFrame, plan: SheetAnalysisPlan) -> tuple[str, ProfilerReport]:
    """
    Asynchronously profiles a single sheet by offloading CPU-bound work to a thread.
    """
    # 1. Prepare the data (drop excluded columns)
    cols_to_drop = list(set(plan.id_like_columns + plan.excluded_from_profiling))
    df_clean = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")

    # 2. RUN PROFILING IN A THREAD (The Fix for Bug #2)
    # This prevents blocking the event loop during heavy statistical analysis
    report = await asyncio.to_thread(get_full_profile, df_clean, skip_discrete_outliers=True)

    # 3. Handle correlations if requested
    if plan.correlation_columns:
        valid = [c for c in plan.correlation_columns if c in df_clean.columns]
        if len(valid) >= 2:
            from typing import cast
            # Explicitly cast to DataFrame to resolve linter confusion with Series.corr
            target_df = cast(pd.DataFrame, df_clean[valid])
            
            # The linter now recognizes this as DataFrame.corr(numeric_only=True)
            report.correlations = await asyncio.to_thread(
                lambda: target_df.corr(numeric_only=True).to_dict()
            )

    return plan.sheet_name, report

async def execute_analysis_mission(file_path: str, target_sheet: str = "order_items") -> dict | None:
    """
    Executes a single-sheet analysis mission (Discovery → Strategy → Profiling → Synthesis).
    Resolves the file path at the start (WSL/Windows and relative paths); uses the resolved path for all file operations.
    Handles .xlsx (with sheet selection) and .csv (no sheet concept).
    """
    path = resolve_file_path(file_path)
    extension = path.suffix.lower()

    # Fail fast if the file does not exist
    if not path.exists():
        logfire.error("File not found: {path}", path=str(path))
        print(f"❌ Error: File not found at {path}")
        return None

    combined_data = {}    

    # Use path.name to keep the logfire span clean
    with logfire.span("Analysis Mission: {file}", file=path.name):

        # Phase 1: Discovery — resolve data structure based on file type
        with logfire.span("Phase 1: Discovery"):
            discovery_map = run_dataset_discovery(path)

            if extension == ".csv":
                # CSV has a single virtual "sheet" keyed by the filename stem
                sheet_key = path.stem
                sheet_meta = discovery_map.sheets.get(sheet_key) or next(iter(discovery_map.sheets.values()), None)
            else:
                # XLSX: look up by the explicitly provided sheet name
                sheet_meta = discovery_map.sheets.get(target_sheet)

            if not sheet_meta:
                logfire.error("Sheet/source '{sheet}' not found", sheet=target_sheet)
                return None

        # Phase 2: Strategic Planning (CDO)
        with logfire.span("Phase 2: Strategic Planning") as span:
            # CDO defines the roadmap and identifies ID columns to ignore
            strategy_res = await orchestrator_agent.run(
                f"Define strategy for {sheet_meta.name}: {sheet_meta.model_dump_json()}"
            )
            strategy = strategy_res.output
            span.set_attribute("hypothesis", strategy.business_hypothesis)
            print(f"🎯 CDO Hypothesis: {strategy.business_hypothesis}")

        # Phase 3: Tactical Execution (Profiling)
        with logfire.span("Phase 3: Tactical Execution"):
            # Load data dynamically based on detected file extension
            if extension == ".csv":
                df = pd.read_csv(path)
            else:
                df = pd.read_excel(path, sheet_name=target_sheet)

            # Strict column exclusion based on CDO instructions to prevent ID hallucinations
            cols_to_drop = list(set(strategy.id_like_columns + strategy.excluded_from_profiling))
            df_clean = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")

            # Ground Truth Generation
            raw_results = get_full_profile(df_clean)

            # Override correlations using the CDO's specific target columns (need 2+ cols for a matrix)
            if hasattr(strategy, "correlation_columns") and strategy.correlation_columns:
                valid_corr_cols = [c for c in strategy.correlation_columns if c in df_clean.columns]
                if len(valid_corr_cols) >= 2:
                    corr_only_df = cast(pd.DataFrame, df_clean[valid_corr_cols])
                    corr_matrix = corr_only_df.corr(numeric_only=True).to_dict()
                    raw_results.correlations = corr_matrix

        # Phase 4: Synthesis + Critic Loop (Max 2 Attempts)
        with logfire.span("Phase 4: Synthesis & Validation"):
            revision_hint = ""
            final_narrative = ""
            final_report_obj = None
            critic_report = None

            for attempt in range(2):
                with logfire.span(f"Attempt {attempt + 1}"):
                    # 1. Analyst generates structured narrative, reasoning, and hypothesis validation
                    mission_prompt = load_prompt("analyst_mission").format(
                        hypothesis=strategy.business_hypothesis,
                        questions="\n".join(f"- {q}" for q in strategy.primary_questions),
                        data=json.dumps(combined_data, indent=2),
                        analysis_mode="separate",
                        cross_sheet_hypothesis="N/A"
                    )

                    if revision_hint:
                        mission_prompt += f"\n\nCRITICAL REVISION REQUIRED:\n{revision_hint}"

                    report_res = await analyst_agent.run(mission_prompt)
                    final_report_obj = report_res.output
                    final_narrative = final_report_obj.final_report_markdown

                    # 2. Critic validates the narrative against ground truth
                    critic_input = (
                        f"PROFILER_DATA:\n{raw_results.model_dump_json()}\n\n"
                        f"ANALYST_NARRATIVE:\n{final_narrative}"
                    )

                    critic_res = await critic_agent.run(critic_input)
                    critic_report = critic_res.output

                    # 3. Decision Gate
                    if critic_report.approved:
                        logfire.info("Report approved by Critic", score=critic_report.score)
                        # Log the internal reasoning chain for observability
                        logfire.info("Analyst CoT", thoughts=final_report_obj.internal_thought_process)
                        print(f"✅ Report Approved (Score: {critic_report.score})")
                        break

                    revision_hint = critic_report.revision_instructions
                    logfire.warn(f"Attempt {attempt + 1} blocked", findings=critic_report.findings)
                    print(f"⚠️ Revision Required (Attempt {attempt + 1}): {revision_hint}")
            else:
                # Runs ONLY if the loop exhausts all attempts without a break
                logfire.warn("Published after max retries", final_score=critic_report.score if critic_report else 0)
                print("🚨 PUBLISHED WITH WARNING: Critic did not fully approve after 2 attempts.")

            print(f"\n{'═'*50}\n📜 FINAL VALIDATED REPORT\n{'═'*50}\n{final_narrative}")

        # Phase 5: Formatting (PowerPoint Generation)
        pptx_path_str = None
        with logfire.span("Phase 5: Formatting"):
            if critic_report and critic_report.approved and final_report_obj:
                # Import here to avoid circular dependencies if defined in another file
                from pptx_renderer import render_pptx
                
                format_prompt = (
                    f"AnalystOutput:\n{final_report_obj.model_dump_json()}\n\n"
                    f"deck_title: {sheet_meta.name} Analysis\n"
                    f"critic_score: {critic_report.score}\n"
                    f"analysis_mode: separate\n"
                )
                
                try:
                    format_res = await formatter_agent.run(format_prompt)
                    out_path = f"output_{path.stem}_{target_sheet}.pptx"
                    pptx_path = render_pptx(format_res.output, out_path)
                    pptx_path_str = str(pptx_path)
                    logfire.info("PPTX generated", path=pptx_path_str)
                    print(f"📊 Presentation saved to: {pptx_path_str}")
                except Exception as e:
                    logfire.error("Failed to generate PPTX", error=str(e))
                    print(f"❌ Failed to generate presentation: {e}")
            else:
                print("⏭️ Skipping PPTX generation because the report was not approved.")
                
        # Update the return dictionary
        return {
            "strategy": strategy,
            "results": raw_results,
            "narrative": final_narrative,
            "hypothesis_validation": final_report_obj.hypothesis_validation if final_report_obj else "",
            "reasoning": final_report_obj.internal_thought_process if final_report_obj else [],
            "critic_score": critic_report.score if critic_report else None,
            "pptx_path": pptx_path_str # Added to return dict
        }

async def execute_multi_sheet_mission(file_path: str) -> dict | None:
    """Execute a multi-sheet analysis mission.

    This Mission coordinates the Phase 3 pipeline:
    Discovery → Strategy (join planning) → join execution → parallel profiling → synthesis.

    Args:
        file_path: User-provided path (relative or absolute). Resolved via `resolve_file_path`.

    Returns:
        A result dict on success, or `None` if the file cannot be found.
    """
    # Use our robust path resolver
    path = resolve_file_path(file_path)
    if not path.exists():
        logfire.error("File not found: {path}", path=str(path))
        return None

    with logfire.span("Multi-Sheet Mission: {file}", file=path.name):

        # Phase 1: Discovery — Detect relationships across all sheets
        with logfire.span("Phase 1: Discovery") as span:
            global_map = run_dataset_discovery(path)
            
            # This is the crucial line for Logfire visibility
            span.set_attribute("discovery_result", global_map.model_dump())
            
            logfire.info("Discovered {n} relationships", n=len(global_map.relationships))

        # Phase 2: CDO Multi-Sheet Strategy — Planning the joins and focus areas
        with logfire.span("Phase 2: Multi-Sheet Strategy") as span:
            strategy_res = await multi_orchestrator_agent.run(
                global_map.model_dump_json(indent=2)
            )
            strategy = strategy_res.output
            span.set_attribute("analysis_mode", strategy.analysis_mode)
            span.set_attribute("joins", len(strategy.joins_to_execute))
            print(f"🎯 Multi-Sheet Hypothesis: {strategy.business_hypothesis}")

        # Phase 3: Data Loading & Joins — Executing CDO's merge instructions
        with logfire.span("Phase 3: Data Loading & Joins"):
            dfs: Dict[str, pd.DataFrame] = {}
            if path.suffix.lower() == ".xlsx":
                for sheet_name in global_map.sheets:
                    dfs[sheet_name] = pd.read_excel(path, sheet_name=sheet_name)
            else:
                dfs[path.stem] = pd.read_csv(path)

            # Execute join instructions to create enriched DataFrames.
            # Each instruction is validated before the merge so schema mismatches
            # produce an informative skip rather than a silent failure.
            for join_instr in strategy.joins_to_execute:
                try:
                    # Guard 1: both source sheets must have been loaded
                    missing_sheets = [
                        s for s in (join_instr.left_sheet, join_instr.right_sheet)
                        if s not in dfs
                    ]
                    if missing_sheets:
                        logfire.warn(
                            "Join '{name}' skipped — sheets not loaded: {sheets}",
                            name=join_instr.result_name,
                            sheets=", ".join(missing_sheets),
                        )
                        print(f"⚠️ Join '{join_instr.result_name}' skipped: sheet(s) not loaded — {', '.join(missing_sheets)}")
                        continue

                    left_df = dfs[join_instr.left_sheet]
                    right_df = dfs[join_instr.right_sheet]

                    # Guard 2: join column must exist in both DataFrames (schema mismatch check)
                    schema_issues = [
                        f"'{join_instr.on_column}' missing in '{sheet}'"
                        for sheet, df in ((join_instr.left_sheet, left_df), (join_instr.right_sheet, right_df))
                        if join_instr.on_column not in df.columns
                    ]
                    if schema_issues:
                        logfire.warn(
                            "Join '{name}' skipped — schema mismatch: {issues}",
                            name=join_instr.result_name,
                            issues="; ".join(schema_issues),
                        )
                        print(f"⚠️ Join '{join_instr.result_name}' skipped — schema mismatch: {'; '.join(schema_issues)}")
                        continue

                    merged = left_df.merge(right_df, on=join_instr.on_column, how=join_instr.join_type)
                    dfs[join_instr.result_name] = merged
                    logfire.info(
                        "Join created: {name} ({rows} rows)",
                        name=join_instr.result_name,
                        rows=len(merged),
                    )
                except Exception as e:
                    # Catch unexpected runtime errors (e.g. dtype incompatibility) without
                    # aborting the entire mission — log and skip this join only.
                    logfire.error(
                        "Join '{name}' failed unexpectedly: {err}",
                        name=join_instr.result_name,
                        err=str(e),
                    )
                    print(f"❌ Unexpected join error for '{join_instr.result_name}': {e}")

        # Phase 4: Parallel Profiling — Concurrent execution using asyncio.gather
        with logfire.span("Phase 4: Parallel Profiling"):
            active_plans = [p for p in strategy.per_sheet_plans if p.should_profile]
            tasks = [
                _profile_sheet_async(dfs[p.sheet_name], p)
                for p in active_plans
                if p.sheet_name in dfs
            ]
            
            # Map results to sheet names
            profile_results_list = await asyncio.gather(*tasks)
            profile_results: Dict[str, ProfilerReport] = dict(profile_results_list)

        # Phase 5: Synthesis & Validation Loop (Max 2 Attempts)
        with logfire.span("Phase 5: Synthesis & Validation"):
            combined_data = {k: v.model_dump() for k, v in profile_results.items()}
            revision_hint = ""
            final_report_obj = None
            critic_report = None

            for attempt in range(2):
                with logfire.span(f"Attempt {attempt + 1}"):
                    # 1. Analyst Synthesis
                    mission_prompt = load_prompt("analyst_mission").format(
                        hypothesis=strategy.business_hypothesis,
                        analysis_mode=strategy.analysis_mode,
                        cross_sheet_hypothesis=strategy.cross_sheet_hypothesis,
                        questions="\n".join(f"- {q}" for q in strategy.primary_questions),
                        data=json.dumps(combined_data, indent=2)
                    )

                    if revision_hint:
                        mission_prompt += f"\n\nCRITICAL REVISION REQUIRED:\n{revision_hint}"

                    report_res = await analyst_agent.run(mission_prompt)
                    final_report_obj = report_res.output

                    # 2. Critic Audit
                    critic_input = (
                        f"PROFILER_DATA:\n{json.dumps(combined_data)}\n\n"
                        f"ANALYST_NARRATIVE:\n{final_report_obj.final_report_markdown}"
                    )
                    critic_res = await critic_agent.run(critic_input)
                    critic_report = critic_res.output

                    # 3. Decision Gate
                    if critic_report.approved:
                        logfire.info("Multi-sheet report approved", score=critic_report.score)
                        print(f"✅ Report Approved (Score: {critic_report.score})")
                        break
                    
                    revision_hint = critic_report.revision_instructions
                    logfire.warn(f"Attempt {attempt + 1} blocked", findings=critic_report.findings)
                    print(f"⚠️ Revision Required (Attempt {attempt + 1}): {revision_hint}")

            print(f"\n{'═'*50}\n📜 FINAL VALIDATED MULTI-SHEET REPORT\n{'═'*50}\n{final_report_obj.final_report_markdown}")

        # Phase 6: Formatting (PowerPoint Generation for Multi-Sheet)
        pptx_path_str = None
        with logfire.span("Phase 6: Formatting"):
            if critic_report and critic_report.approved and final_report_obj:
                # Import here to avoid circular dependencies
                from pptx_renderer import render_pptx
                
                format_prompt = (
                    f"AnalystOutput:\n{final_report_obj.model_dump_json()}\n\n"
                    f"deck_title: {path.stem} Comprehensive Analysis\n"
                    f"critic_score: {critic_report.score}\n"
                    f"analysis_mode: {strategy.analysis_mode}\n"
                )
                
                try:
                    format_res = await formatter_agent.run(format_prompt)
                    out_path = f"output_{path.stem}_multisheet.pptx"
                    pptx_path = render_pptx(format_res.output, out_path)
                    pptx_path_str = str(pptx_path)
                    logfire.info("PPTX generated", path=pptx_path_str)
                    print(f"📊 Presentation saved to: {pptx_path_str}")
                except Exception as e:
                    logfire.error("Failed to generate PPTX", error=str(e))
                    print(f"❌ Failed to generate presentation: {e}")
            else:
                print("⏭️ Skipping PPTX generation because the multi-sheet report was not approved.")
                
        # Update the return dictionary
        return {
            "strategy": strategy,
            "profile_results": profile_results,
            "narrative": final_report_obj.final_report_markdown if final_report_obj else "",
            "critic_score": critic_report.score if critic_report else 0.0,
            "reasoning": final_report_obj.internal_thought_process if final_report_obj else [],
            "pptx_path": pptx_path_str # Added to return dict
        }

# Public API: mission entry points only (no private helpers)
__all__ = ["execute_analysis_mission", "execute_multi_sheet_mission"]


In [ ]:
# cell dd3f2ac7
from IPython.display import display, Markdown
import json

# 1. Configuration
target_file_name = "titanic_data.xlsx"
target_sheet_name = "titanic_data" # Ensure this matches the actual sheet name

# 2. Execute the mission
# Note: Ensure execute_analysis_mission returns 'strategy' and 'critic_score'
mission_data = await execute_analysis_mission(
    file_path=target_file_name,
    target_sheet=target_sheet_name
)

# 3. Render Results
if mission_data and "narrative" in mission_data:
    score = mission_data.get("critic_score", 0.0)
    display(Markdown("---"))

    # Audit Status Header (Strict Scoring)
    if score == 0.0:
        display(Markdown(f"### 🛑 AUDIT FAILED — BLOCKED BY CRITIC (Score: {score}/1.0)"))
    elif score < 0.8:
        display(Markdown(f"### ⚠️ APPROVED WITH WARNINGS (Score: {score}/1.0)"))
    else:
        display(Markdown(f"### ✅ EXCELLENT — FULLY APPROVED (Score: {score}/1.0)"))

    # --- Section A: Strategic Context (The 'Why') ---
    # Added this to show what the CDO actually planned before the validation
    strategy = mission_data.get("strategy")
    if strategy:
        display(Markdown("#### 🎯 Strategic Goal (CDO)"))
        display(Markdown(f"**Hypothesis:** {strategy.business_hypothesis}"))
        display(Markdown("---"))

    # --- Section B: Hypothesis Validation ---
    hyp_validation = mission_data.get("hypothesis_validation", "")
    if hyp_validation:
        display(Markdown("#### 🔬 Analyst Validation Result"))
        display(Markdown(hyp_validation))
        display(Markdown("---"))

    # --- Section C: Chain-of-Thought ---
    reasoning_steps = mission_data.get("reasoning", [])
    if reasoning_steps:
        reasoning_md = "#### 🧠 Internal Reasoning (Chain-of-Thought)\n\n"
        for step in reasoning_steps:
            reasoning_md += f"- {step}\n"
        display(Markdown(reasoning_md))
        display(Markdown("---"))

    # --- Section D: Final Validated Report ---
    display(Markdown("#### 📜 Final Validated Report"))
    display(Markdown(mission_data["narrative"]))
    display(Markdown("---"))

else:
    # Improved error feedback
    print(f"❌ Mission failed for {target_file_name}.")
    print("Check if the file exists and if the return keys match the display requirements.")

✅ Found file at: /mnt/c/Users/grosp/Downloads/titanic_data.xlsx
05:02:51.271 Analysis Mission: titanic_data.xlsx
05:02:51.313   Phase 1: Discovery
05:02:51.895   Phase 2: Strategic Planning
🎯 CDO Hypothesis: Passenger survival on the Titanic was influenced by socio-economic factors (class, fare) and demographic characteristics (age, gender), with wealthy passengers having a higher survival rate.
05:03:04.186   Phase 3: Tactical Execution
05:03:06.636   Phase 4: Synthesis & Validation
05:03:06.637     Attempt 1
05:03:27.189       Report approved by Critic
05:03:27.190       Analyst CoT
✅ Report Approved (Score: 0.85)

══════════════════════════════════════════════════
📜 FINAL VALIDATED REPORT
══════════════════════════════════════════════════
## Key Findings
### Survival by Passenger Class
- **Finding:** There is a negative correlation between passenger class and survival (r=-0.338). This suggests that passengers in higher classes (1st class) had better survival rates compared to those 

---

### ✅ EXCELLENT — FULLY APPROVED (Score: 0.85/1.0)

#### 🔬 Hypothesis Validation

The hypothesis is PARTIALLY CONFIRMED because there is a notable negative correlation between Pclass and survival (r=-0.338), and a positive correlation between Fare and survival (r=0.257). However, the correlations for age (r=-0.077), SibSp (r=-0.035), and Parch (r=0.082) indicate those factors had a minimal influence. The lack of data on gender prevents complete validation.

...

#### 🧠 Analyst's Internal Reasoning (Chain-of-Thought)

- Survival rate by passenger class shows a negative correlation with Pclass (r=-0.338) indicating higher class passengers were more likely to survive.
- Gender analysis is absent in the data but typically, gender (encoded as 'Sex') would need correlation analysis with survival rates.
- Age and survival show a very weak negative correlation (r=-0.077), suggesting minimal age bias in survival rates.
- Fare correlates positively with survival (r=0.257), implying wealthier passengers had better survival odds.
- Family size (proxy by 'SibSp' and 'Parch') shows weak correlations, with SibSp negatively correlated to survival (r=-0.035) and Parch positively (r=0.082).


---

#### 📜 Final Validated Report

## Key Findings
### Survival by Passenger Class
- **Finding:** There is a negative correlation between passenger class and survival (r=-0.338). This suggests that passengers in higher classes (1st class) had better survival rates compared to those in lower classes (3rd class).

### Business Implication
- **Implication:** This supports the hypothesis that socio-economic status, as represented by passenger class, influenced survival rates.

### Recommended Action
- **Action:** Insights can refine historical narratives regarding the Titanic, focusing on class-driven survival advantages.


### Impact of Wealth (Fare) on Survival
- **Finding:** A positive correlation between fare and survival (r=0.257) indicates that passengers who paid higher fares had better survival chances.

### Business Implication
- **Implication:** Wealthier individuals were prioritized or had more access to survival resources like lifeboats.

### Recommended Action
- **Action:** Future studies or exhibitions should highlight the economic factors of this historical event, possibly influencing current safety policies.


### Age and Family Size
- **Finding:** There is a weak correlation between age (r=-0.077), SibSp (r=-0.035), and Parch (r=0.082) with survival, suggesting minimal influence on outcomes.

### Business Implication
- **Implication:** Rejects the premise that age and family size were major factors in survival, allowing focus on socio-economic factors.

### Recommended Action
- **Action:** Direct research efforts towards socio-economic disparities rather than demographic factors when considering safety and survival strategies.

---

In [9]:
# cell dd3f2ac7
from IPython.display import display, Markdown
import json

# 1. Target file name
target_file = "bikes company.xlsx"

# 2. Execute the Phase 3 Multi-Sheet Mission
# Ensure execute_multi_sheet_mission returns strategy, critic_score, reasoning, and narrative
mission_data = await execute_multi_sheet_mission(
    file_path=target_file
)

# 3. Render the formatted results
if mission_data and "narrative" in mission_data:
    score = mission_data.get("critic_score", 0.0)
    display(Markdown("---"))

    # Audit Status Header (Strict Scoring)
    if score == 0.0:
        display(Markdown(f"### 🛑 MULTI-SHEET AUDIT FAILED (Score: {score}/1.0)"))
    elif score < 0.8:
        display(Markdown(f"### ⚠️ APPROVED WITH WARNINGS (Score: {score}/1.0)"))
    else:
        display(Markdown(f"### ✅ EXCELLENT — FULLY APPROVED (Score: {score}/1.0)"))

    # Section A: Strategic Context (CDO Planning)
    strategy = mission_data.get("strategy")
    if strategy:
        display(Markdown("#### 🎯 Strategic Context"))
        # Added Analysis Mode and Cross-Sheet Hypothesis for full context
        display(Markdown(f"**Mode:** `{strategy.analysis_mode}`"))
        display(Markdown(f"**Main Hypothesis:** {strategy.business_hypothesis}"))
        if hasattr(strategy, 'cross_sheet_hypothesis') and strategy.cross_sheet_hypothesis:
             display(Markdown(f"**Cross-Sheet Hypothesis:** {strategy.cross_sheet_hypothesis}"))
        display(Markdown("---"))

    # Section B: Analyst Chain-of-Thought
    reasoning_steps = mission_data.get("reasoning", [])
    if reasoning_steps:
        reasoning_md = "#### 🧠 Analyst's Internal Reasoning (Chain-of-Thought)\n\n"
        for step in reasoning_steps:
            reasoning_md += f"- {step}\n"
        display(Markdown(reasoning_md))
        display(Markdown("---"))

    # Section C: Final Validated Multi-Sheet Report
    display(Markdown("#### 📜 Final Validated Report"))
    display(Markdown(mission_data["narrative"]))
    display(Markdown("---"))

else:
    print(f"❌ Mission failed for '{target_file}'. Check if the file exists or if the return keys match.")

✅ Found file at: /mnt/c/Users/grosp/Downloads/bikes company.xlsx
23:49:54.600 Multi-Sheet Mission: bikes company.xlsx
23:49:54.600   Phase 1: Discovery
23:49:57.976     Discovered 18 relationships
23:49:57.979   Phase 2: Multi-Sheet Strategy
🎯 Multi-Sheet Hypothesis: The company aims to analyze sales performance, inventory management, and customer behavior across different stores to optimize operations and increase revenue.
23:50:20.406   Phase 3: Data Loading & Joins
23:50:22.129     Join created: orders_with_items (4722 rows)
23:50:22.130     Join created: product_details (321 rows)
23:50:22.131     Join created: customer_orders (1615 rows)
23:50:22.132   Phase 4: Parallel Profiling
23:50:22.156   Phase 5: Synthesis & Validation
23:50:22.156     Attempt 1
23:50:47.157       Multi-sheet report approved
✅ Report Approved (Score: 0.85)

══════════════════════════════════════════════════
📜 FINAL VALIDATED MULTI-SHEET REPORT
══════════════════════════════════════════════════
## Key Findin

---

### ✅ EXCELLENT — FULLY APPROVED (Score: 0.85/1.0)

#### 🎯 Strategic Context

**Mode:** `both`

...

#### 🧠 Analyst's Internal Reasoning (Chain-of-Thought)

- Correlation between list price and quantity is -0.0014, indicating a negligible relationship; changes in list prices do not influence quantity sold substantially.
- 422 price outliers in order_items suggest high-end products at $3999.99; also observed in product and orders_with_items data sets.
- 10.76% missing shipped dates in orders_with_items limit timeline analysis for sales completion.
- Phone missing in 87% of customer_orders, affecting targeted marketing potential based on contact information.


---

#### 📜 Final Validated Report

## Key Finding
- The data highlights a minimal impact of list price on the quantity sold with a correlation of -0.0014, suggesting that price adjustments are unlikely to strongly affect sales volume. Moreover, a small concentration of high-end products (sample $3999.99) should be noted for targeted marketing.

## Business Implication
- Pricing strategies might not significantly influence sales volume based on current data. However, marketing efforts could focus more on premium segments where price segmentation is evident. 

## Recommended Action
- Marketing campaigns should highlight the premium products, possibly developing exclusive promotions around them to attract high-value customers. More accurate customer targeting might be constrained by missing phone data in 87% of customer records, requiring improved data collection strategies.

---